# GLiNER large -- PII/PHI multi-label eval

Loads `urchade/gliner_large-v2.1`, runs it over `data/sample_docs.jsonl` with
per-label thresholds (standard labels + your own zero-shot prompts), and
reports precision / recall (sensitivity) / F1 per label plus micro & macro.

In [ ]:
import json
import sys

import pandas as pd
from gliner import GLiNER

sys.path.append("../src")
from labels import LABEL_THRESHOLDS
from metrics import evaluate

In [ ]:
model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")
LABELS = list(LABEL_THRESHOLDS)
LABEL_THRESHOLDS

In [ ]:
with open("../data/sample_docs.jsonl") as f:
    docs = [json.loads(line) for line in f]
len(docs)

In [ ]:
def predict_with_thresholds(text: str) -> list[dict]:
    """Run GLiNER once at the lowest configured threshold, then apply each
    label's own threshold in a second pass (predict_entities only takes one
    global threshold). A prediction that clears no label's threshold simply
    isn't returned -- that IS the "non-PII" case, no extra label needed."""
    floor = min(LABEL_THRESHOLDS.values())
    raw = model.predict_entities(text, LABELS, threshold=floor, multi_label=True)
    return [e for e in raw if e["score"] >= LABEL_THRESHOLDS[e["label"]]]


predictions = [predict_with_thresholds(d["text"]) for d in docs]
for d, preds in zip(docs, predictions):
    print(d["text"][:60], "...")
    for p in preds:
        print(f"  {p['label']:35s} {p['score']:.2f}  {p['text']!r}")

In [ ]:
gold_docs = [d["entities"] for d in docs]
rows, micro, macro = evaluate(gold_docs, predictions, LABELS)

per_label_df = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
per_label_df

In [ ]:
print("micro:", micro)  # dominated by frequent labels
print("macro:", macro)  # every label weighted equally -- watch this for rare PHI labels

In [ ]:
# Document-level "did we catch that this doc contains ANY PII" sensitivity.
# In a redaction pipeline, missing a whole document is worse than missing
# one span inside a document that was otherwise flagged.
doc_hits = sum(1 for g, p in zip(gold_docs, predictions) if not g or len(p) > 0)
positive_docs = sum(1 for g in gold_docs if g)
true_positive_docs = sum(1 for g, p in zip(gold_docs, predictions) if g and len(p) > 0)
print(f"document-level PII recall (sensitivity): {true_positive_docs}/{positive_docs}")